# Metric and Target Generation

Create out-of-fold survival targets used by downstream regression models.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.config import CFG
from src.modeling import MD
from src.preprocessing import get_categorical_columns, load_training_data, split_dataset

In [3]:
dataframe = load_training_data(CFG.train_path)
train_data, validation_data, test_data = split_dataset(
    dataframe,
    validation_size=CFG.validation_size,
    test_size=CFG.test_size,
    random_state=CFG.random_state,
    stratify_column=CFG.split_stratify_column,
)
cat_cols = get_categorical_columns(train_data)
md = MD(CFG.color, train_data, cat_cols, CFG.early_stop, CFG.penalizer, CFG.n_splits, CFG.random_state)

## Split analysis
The workflow now uses only the original training file. The validation and test sets are internal holdout splits, so their labels remain available for evaluation.


In [4]:
pip install lifelines

Note: you may need to restart the kernel to use updated packages.


In [5]:
train_data = md.create_targets()
display(train_data[["ID", "cox_hazard", "km_survival", "na_hazard", "event_time"]].head())

c:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.


c:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.


c:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column b

Overall Stratified C-Index Score for Cox: 0.6551
Overall Stratified C-Index Score for Kaplan-Meier: 0.9987
Overall Stratified C-Index Score for Nelson-Aalen: 0.9987


,ID,cox_hazard,km_survival,na_hazard,event_time
0,22706,2.247843,0.957755,-0.043162,3.157000
1,22166,2.580313,0.852427,-0.159663,4.628000
2,7943,0.773400,0.463031,-0.769927,19.618999
3,3110,0.418315,0.460180,-0.776103,-24.423000
4,27485,0.799119,0.459773,-0.776987,-27.555000


## Target analysis
The generated targets capture survival information from different estimators. Cox hazard is the main risk-style target, while Kaplan-Meier and Nelson-Aalen add complementary survival and cumulative-hazard views.
